# Notebook 2 — Preprocessing & Feature Engineering

## Yapılacaklar

### 1. Veri yükleme
- [ ] NB1'den işlenmiş df'i yükle (veya raw CSV'den başla, aynı temizliği uygula)
- [ ] `_c39` kolonunu drop et
- [ ] `"?"` → NaN dönüşümü (collision_type, property_damage, police_report_available)
- [ ] `fraud_reported` → 0/1 map

### 2. Feature Engineering
- [ ] `vehicle_age` = incident_year - auto_year
- [ ] `injury_ratio` = injury_claim / total_claim_amount
- [ ] `is_single_vehicle` = incident_type == 'Single Vehicle Collision' → 1/0
- [ ] `is_major_damage` = incident_severity == 'Major Damage' → 1/0
- [ ] `no_witness_no_police` = witnesses==0 & police_report_available=='NO' → 1/0
- [ ] `policy_age` = incident_year - policy_bind_year

### 3. Encoding
- [ ] Low cardinality object kolonlar → Label Encoding
- [ ] High cardinality kolonlar → drop veya target encoding karar ver
- [ ] `policy_number`, `insured_zip`, `incident_location` → drop

### 4. Missing value imputation
- [ ] Numeric → median
- [ ] Kategorik → mode veya 'Unknown'

### 5. Train/test split
- [ ] StratifiedKFold için hazırlık
- [ ] preprocessor.pkl kaydet
- [ ] features.pkl kaydet

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import joblib


df = pd.read_csv('insurance_claims.csv')

df['incident_year'] = pd.to_datetime(df['incident_date']).dt.year
df['policy_bind_year'] = pd.to_datetime(df['policy_bind_date']).dt.year


drop_cols = ['policy_number', 'insured_zip', 'incident_location', 'incident_date', 'policy_bind_date', '_c39', 'auto_model', 'incident_date']



df = df.drop(columns=drop_cols)
df[['property_damage', 'police_report_available', 'collision_type']] = df[['property_damage', 'police_report_available', 'collision_type']].replace('?', np.nan)

df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})

#Feature Engineering



#--------------------------------------

cat_cols = df.select_dtypes(include='object').columns.tolist()

X = df.drop(columns='fraud_reported')
y = df['fraud_reported']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder())
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', cat_pipeline, cat_cols)
], remainder='passthrough')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
])

X_train_processed = pipeline.fit_transform(X_train, y_train)
X_test_processed = pipeline.transform(X_test)

joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(X_train_processed, 'X_train_processed.pkl')
joblib.dump(X_test_processed, 'X_test_processed.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test, 'y_test.pkl')

print(type(X_train))

defaults = X_train.mode().iloc[0]
joblib.dump(defaults, 'defaults.pkl')



<class 'pandas.DataFrame'>


C:\Users\User\AppData\Local\Temp\ipykernel_12056\2901469381.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


['defaults.pkl']

df['vehicle_age'] = df['incident_year'] - df['auto_year']
df['policy_age'] = df['incident_year'] - df['policy_bind_year']
df['injury_ratio'] = df['injury_claim'] / df['total_claim_amount']
df['is_single_vehicle'] = (df['incident_type'] == 'Single Vehicle Collision').astype(int)
df['is_major_damage'] = (df['incident_severity'] == 'Major Damage').astype(int)
df['no_witness_no_police'] = ((df['witnesses'] == 0) & (df['police_report_available'] == 'NO')).astype(int)